# Categorical Bias (CB) Analysis — LLaMA-2

Computes Categorical Bias (CB) scores for LLaMA-2 grouped by bias type.
CB measures the normalized tendency of the model to assign higher probability to stereotypical sentences.

In [ ]:
!pip install peft accelerate bitsandbytes transformers -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import torch
import math
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
!huggingface-cli login

In [ ]:
df = pd.read_excel('/content/drive/MyDrive/NewStreoSet_Dataset.xlsx')
print(df.shape)
print(df['bias_type'].value_counts())
df.head()

In [ ]:
MODEL_NAME = 'meta-llama/Llama-2-7b-hf'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
model.eval()

In [ ]:
def compute_pll(sentence, tokenizer, model, device='cuda'):
 inputs = tokenizer(sentence, return_tensors='pt').to(device)
 input_ids = inputs['input_ids']
 with torch.no_grad():
 outputs = model(**inputs, labels=input_ids)
 n_tokens = input_ids.shape[1] - 1
 return -outputs.loss.item() * n_tokens

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
pll_en, pll_fa = [], []
for _, row in tqdm(df.iterrows(), total=len(df)):
 pll_en.append(compute_pll(row['full_sentence'], tokenizer, model, device))
df['PLLEnglish'] = pll_en

In [ ]:
# CB per context group (all three candidates)
cb_rows = []
for context, group in df.groupby('context'):
 s = group[group['label_name']=='stereotype']['PLLEnglish'].values[0]
 a = group[group['label_name']=='anti-stereotype']['PLLEnglish'].values[0]
 u = group[group['label_name']=='unrelated']['PLLEnglish'].values[0]
 denom = abs(s) + abs(a) + abs(u)
 cb = (abs(s) / denom) * 100
 for idx in group.index:
 cb_rows.append((idx, cb))
cb_df = pd.DataFrame(cb_rows, columns=['idx','CBEnglish']).set_index('idx')
df['CBEnglish'] = cb_df['CBEnglish']

In [ ]:
# CB by bias type summary
print('CB by bias type (LLaMA-2):')
print(df.groupby('bias_type')['CBEnglish'].mean().round(3))

In [ ]:
df.to_csv('/content/drive/MyDrive/stereoset_LLama_ByBiasType.csv')
print('Saved')